In [15]:
!pip install bangla-stemmer


  Using cached bangla_stemmer-1.0-py3-none-any.whl.metadata (2.4 kB)
Using cached bangla_stemmer-1.0-py3-none-any.whl (9.1 kB)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
!pip install imbalanced-learn

  Using cached imbalanced_learn-0.14.0-py3-none-any.whl.metadata (8.8 kB)
Using cached imbalanced_learn-0.14.0-py3-none-any.whl (239 kB)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
import numpy as np
import pandas as pd
newdata = pd.read_csv("../data/final_bn_data.csv")
newdata.head()

,category,headline,content,label
0,National,৮ দিনে ১৮ বিল পাস!,দশম জাতীয় সংসদের মেয়াদ শেষ হয়ে যাচ্ছে। কার্যত ...,0.0
1,Sports,আ’লীগের জনসভায় লোকে লোকারণ্য ফেনী ট্রাংক রোড,একাদশ জাতীয় সংসদ নির্বাচনকে সামনে রেখে সাংগঠনি...,0.0
2,National,মাদ্রাসায় জোড়া খুন: পরিচালক তিন দিনের রিমান্ডে,গাজীপুরে জোড়া খুন মামলার প্রধান আসামি মাদ্রাসা...,0.0
3,Sports,নেপালকে হারিয়ে গ্রুপ চ্যাম্পিয়ন বাংলাদেশ,সাফ অনূর্ধ্ব-১৮ নারী ফুটবল চ্যাম্পিয়নশিপে নেপা...,1.0
4,National,কুড়িগ্রামে ২ শিক্ষার্থীর লাশ উদ্ধার,কুড়িগ্রাম প্রতিনিধি : কুড়িগ্রাম সদর উপজেলার বে...,1.0


In [53]:
newdata= newdata.drop(['headline', 'category'], axis=1)
newdata.to_csv('newDataset.csv', index=False)


In [54]:
# Install required package
!pip install bangla-stemmer  # CHANGED: kept same

import pandas as pd
import re
import string
from bangla_stemmer.stemmer import stemmer

# Bangla stopwords list (keep your full list)
bangla_stopwords = set([  # CHANGED: converted to set for faster lookup
    'অবশ্য','অনেক','অনেকে','অনেকেই','অন্তত','অন্য','আজ','আছে','আপনার','আপনি',
'আবার','আমরা','আমাকে','আমাদের','আমার','আমি','আরও','আর','আগে','আগেই','আই',
'অতএব','আগামী','অবধি','অনুযায়ী','আদ্যভাগে','এই','একই','একে','একটি','এখন',
'এখনও','এখানে','এখানেই','এটি','এটা','এটাই','এতটাই','এবং','একবার','এবার',
'এদের','এঁদের','এমন','এমনকী','এল','এর','এরা','এঁরা','এস','এতে','এসে','এ',
'ঐ','ই','ইহা','ইত্যাদি','উনি','উপর','উপরে','উচিত','ও','ওই','ওর','ওরা','ওঁর',
'ওঁরা','ওকে','ওদের','ওঁদের','ওখানে','কত','কবে','করতে','কয়েক','কয়েকটি','করবে',
'করলেন','করার','কারও','করা','করি','করিয়ে','করাই','করলে','করিতে','করিয়া','করেছিলেন',
'করছে','করছেন','করেছেন','করেছে','করেন','করবেন','করায়','করে','করেই','কাছ','কাছে',
'কাজে','কারণ','কিছু','কিছুই','কিন্তু','কিংবা','কি','কী','কেউ','কেউই','কাউকে','কেন',
'কে','কোনও','কোনো','কোন','কখনও','ক্ষেত্রে','খুব','গুলি','গিয়ে','গিয়েছে','গেছে',
'গেল','গেলে','গোটা','চলে','ছাড়া','ছাড়াও','ছিলেন','ছিল','জন্য','জানা','ঠিক','তিনি',
'তিনঐ','তিনিও','তখন','তবে','তবু','তাঁদের','তাঁরা','তাঁর','তাঁকে','তাই','তেমন','তাকে',
'তাহা','তাহাতে','তাহার','তাদের','তারপর','তারা','তার','তাহলে','তা','তাও','তাতে','তো',
'তত','তুমি','তোমার','তথা','থাকে','থাকা','থাকায়','থেকে','থেকেও','থাকবে','থাকেন',
'থাকবেন','থেকেই','দিকে','দিতে','দিয়ে','দিয়েছে','দিয়েছেন','দিলেন','দু','দুটি','দুটো',
'দেয়','দেওয়া','দেওয়ার','দেখা','দেখে','দেখতে','দ্বারা','ধরে','ধরা','নয়','নানা','না',
'নাকি','নাগাদ','নিতে','নিজে','নিজেই','নিজের','নিজেদের','নিয়ে','নেওয়া','নেওয়ার','নেই',
'নাই','পক্ষে','পর্যন্ত','পাওয়া','পারেন','পারি','পারে','পরে','পরেই','পরেও','পর','পেয়ে',
'প্রতি','প্রভৃতি','প্রায়','ফের','ফলে','ব্যবহার','বলতে','বললেন','বলেছেন','বলল','বলা',
'বলেন','বলে','বহু','বসে','বার','বা','বিনা','বরং','বদলে','বাদে','বিশেষ','বিভিন্ন',
'বিষয়টি','ব্যাপারে','ভাবে','ভাবেই','মধ্যে','মধ্যেই','মধ্যেও','মধ্যভাগে','মাধ্যমে','মাত্র',
'মতো','মতোই','মোটেই','যখন','যদি','যদিও','যাবে','যায়','যাকে','যাওয়া','যাওয়ার','যত',
'যতটা','যা','যার','যারা','যাঁর','যাঁরা','যাদের','যান','যাচ্ছে','যেতে','যাতে','যেন','যেমন',
'যেখানে','যিনি','যে','রেখে','রাখা','রয়েছে','রকম','শুধু','সঙ্গে','সঙ্গেও','সমস্ত','সব',
'সবার','সহ','সুতরাং','সহিত','সেই','সেটা','সেটি','সেটাই','সেটাও','সম্প্রতি','সেখান',
'সেখানে','সে','স্পষ্ট','স্বয়ং','হইতে','হইবে','হৈলে','হইয়া','হচ্ছে','হত','হতে','হতেই',
'হবে','হবেন','হয়েছিল','হয়েছে','হয়েছেন','হয়ে','হয়নি','হয়','হয়েই','হয়তো','হল','হলে',
'হলেই','হলেও','হলো','হিসাবে','হওয়া','হওয়ার','হওয়ায়','হন','হোক','জন','জনকে','জনের',
'জানতে','জানায়','জানিয়ে','জানানো','জানিয়েছে','জন্য','জন্যওজে','জে','বেশ','দেন','তুলে',
'ছিলেন','চান','চায়','চেয়ে','মোট','যথেষ্ট','টি','অনুযায়ী','উত্তর','কমনে','প্রথম','প্রযন্ত',
'প্রাথমিক','মনে','নতুন','পি','পেয়্র্','বক্তব্য','ধামার','সি','সাধারণ','সামনে','হাজার'
])

# Simple stemmer for common suffixes (same as your function)
def simple_bangla_stem(word):
    suffixes = ["গুলো", "দের", "মতো", "গুলোই", "টিতে", "টার", "রা", "টি", "টা", "তে", "র", "এর", "ই", "ও"]
    for suf in sorted(suffixes, key=len, reverse=True):
        if word.endswith(suf) and len(word) > len(suf) + 1:
            return word[:-len(suf)]
    return word

# Normalization dictionary (same, just cleaned formatting)
bangla_normalization_dict = {
    "মুরগি": "মুরগি",
    "মুরগী": "মুরগি",
    "মুরগীটা": "মুরগি",
    "মুরগিটা": "মুরগি",
    "মুরগিগুলো": "মুরগি",
    "শিয়াল": "শিয়াল",
    "শেয়াল": "শিয়াল",
    "শিয়াল": "শিয়াল",
    "মানুষ": "মানুষ",
    "মাঞ্জুষ": "মানুষ",
    "মানুষটা": "মানুষ",
    "ভিডি": "ভিডিও",
    "ফরান্স": "ফ্রান্স",
    "খাচ্ছে": "খাওয়া",
    "খাচ্ছেন": "খাওয়া",
    "খায়": "খাওয়া",
    "খেতে": "খাওয়া",
    "খেয়েছে": "খাওয়া",
    "দেয়": "দেওয়া",
    "দিচ্ছে": "দেওয়া",
    "দিয়েছে": "দেওয়া",
    "কিংবা": "কিন্তু",
    "একটি": "এক",
    "দুইটি": "দুই",
    # add more as needed
}

# Suppress print outputs from stemmer
import sys, io
def suppress_stdout(func, *args, **kwargs):
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        result = func(*args, **kwargs)
    finally:
        sys.stdout = old_stdout
    return result

# Main preprocessing function
def preprocessNew_bangla_text(text):
    text = str(text)

    # Remove HTML, URLs, emails, hashtags, Latin letters, and numbers
    text = re.sub(r"<.*?>|http\S+|www\S+|\S*@\S*|\#\S*", " ", text)  # CHANGED: combined into one regex
    text = re.sub(r"[a-zA-Z০-৯0-9]+", " ", text)  # CHANGED: removed unnecessary parts

    # Remove punctuation
    punct_pattern = f"[{string.punctuation}।॥‘’“”—৳]"  # CHANGED: unified punctuation
    text = re.sub(punct_pattern, " ", text)

    # Normalize certain characters
    text = re.sub("[য়য়]", "য়", text)
    text = re.sub("[ঋৃ]", "ঋ", text)
    text = re.sub("[ঁ্]", "", text)
    text = re.sub(r"\s+", " ", text).strip()  # CHANGED: condensed whitespace cleanup

    # Remove consecutive repeated words
    words = text.split()
    words = [words[i] for i in range(len(words)) if i == 0 or words[i] != words[i-1]]  # CHANGED: list comprehension

    # Normalize words using dictionary
    words = [bangla_normalization_dict.get(w, w) for w in words]  # CHANGED: more pythonic

    # Remove stopwords
    words = [w for w in words if w not in bangla_stopwords]  # CHANGED: set lookup faster

    # Apply simple stemming
    words = [simple_bangla_stem(w) for w in words]

    return " ".join(words)

# Load dataset
new_data = pd.read_csv("newDataset.csv")

# Apply preprocessing
new_data['content'] = new_data['content'].apply(preprocessNew_bangla_text)  # CHANGED: applied vectorized apply

# Preview results
for original, processed in zip(new_data['content'][:5], new_data['content'][:5]):  # CHANGED: preview 5 rows instead of 2
    print("Original:", original)
    print("Processed:", processed)
    print("-" * 50)


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Original: দশম জাতীয় সংস মেয়াদ শেষ য়াচছে কারয়ত মাত এক অধিবেশন বরতমান সরকারে অকটোবরে মাঝামাঝি সংকষিপত এক অধিবেশন ডেকে ‍ইিত টানা সংসেদ এরপ ডিসেমবরে শেষ সপতাহে একাদশ সংসদ নিরবাচনে য়া সংখয়াগরিষঠতা পাবেন তারা সরকা গঠন জানুয়ারি বস সরকারে পরথম অধিবেশন বরতমান সরকারে পাচ বছ পূরণ হচছে বছরে জানুয়ারি সংবিধান অনুয়ায়ী বিদয়মান সংসদ ভেঙে আগে দিনে মধয়ে পরবরতী সংসদ নিরবাচন কথা হিসেবে অকটোব নিরবাচনে কাউন ডাউন শুরু সরকারে বিভিনন মহলে আলোচনা পরিপরেকষি ধারণা হচছে অকটোবরে মধয় নিরবাচনকালীন সরকা গঠন সরকা রুটিন ওয়ারক দেশে উদবুদধ পরিসথিতি সঋষ রাষটরপতি চাইলে য়েকোনো সময় অধিবেশন আহবান এদিকে দশম জাতীয় সংস তম অধিবেশন রেকরডে খাতায় লেখা কেননা অধিবেশনে মাত আট কারয়দিবসে মধয়ে ইতোমধয়ে বিল পাস শেষ দিনে চার পাস য়াচছে হিসেবে অননয় উচচতায় অধিবেশন এতো অলপ সময়ে সংখয়ক বিল পাসে নজি গত সেপটেমব শুরু তম অধিবেশন য়া সেপটেমব শেষ য়াচছে শুক শনিবা বাদ টানা কারয়দিবস অধিবেশন পরথম দিন বরতমান সংস সদসয়ে মঋতয়ু শোক পরসতাব গঋহীত অনয়ানয় সকল কারয়সূচি সথগিত তাছাড়া সেপটেমব বিদয়ুৎ বিপরয়য়ে কারণে এক ঘণ অধিবেশন চালানো সথগিত য়ায় কারণে দিন চললে মূলত কারয়দিবসে

In [56]:
class_counts = newdata['label'].value_counts()
print(class_counts)

label
1.0    10000
0.0     4537
Name: count, dtype: int64


In [60]:
new_data = pd.read_csv("newDataset.csv")
newdata.head()

,content,label
0,দশম জাতীয় সংসদের মেয়াদ শেষ হয়ে যাচ্ছে। কার্যত ...,0.0
1,একাদশ জাতীয় সংসদ নির্বাচনকে সামনে রেখে সাংগঠনি...,0.0
2,গাজীপুরে জোড়া খুন মামলার প্রধান আসামি মাদ্রাসা...,0.0
3,সাফ অনূর্ধ্ব-১৮ নারী ফুটবল চ্যাম্পিয়নশিপে নেপা...,1.0
4,কুড়িগ্রাম প্রতিনিধি : কুড়িগ্রাম সদর উপজেলার বে...,1.0


In [ ]:
#logistic regression
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
import numpy as np

# Load dataset
nd = pd.read_csv('newDataset.csv')  
X = nd['content']  
y = nd['label']      


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=20000, 
    ngram_range=(1,3),
    sublinear_tf=True,
    lowercase=True
)
X_train_vect = tfidf.fit_transform(X_train)
X_test_vect = tfidf.transform(X_test)

# Logistic Regression with class balancing
clf = LogisticRegression(class_weight='balanced', max_iter=2000, solver='saga', random_state=42)

# Hyperparameter tuning for C (regularization strength)
param_grid = {'C': [0.1, 0.5, 1, 2, 5]}
grid = GridSearchCV(clf, param_grid, scoring='f1_macro', cv=5, n_jobs=-1)
grid.fit(X_train_vect, y_train)

best_clf = grid.best_estimator_
print(f"Best C: {grid.best_params_['C']}")

# Predict probabilities 
y_prob = best_clf.predict_proba(X_test_vect)[:,1]
threshold = 0.48  
y_pred = (y_prob >= threshold).astype(int)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))



Best C: 5

Test Accuracy: 0.7510

Classification Report:
              precision    recall  f1-score   support

         0.0       0.59      0.64      0.62       908
         1.0       0.83      0.80      0.82      2000

    accuracy                           0.75      2908
   macro avg       0.71      0.72      0.72      2908
weighted avg       0.76      0.75      0.75      2908

Confusion Matrix:
[[ 582  326]
 [ 398 1602]]
